In [2]:
# ============================================================
# MOVIE GENRE CLASSIFICATION
# TF-IDF + NAIVE BAYES + LOGISTIC REGRESSION + SVM
# ============================================================

import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report
)


# ============================================================
# 1. LOAD TRAINING DATA
# ============================================================

train_file = "Genre Classification Dataset/train_data.txt"

train_df = pd.read_csv(
    train_file,
    sep=":::",
    engine="python",
    header=None,
    names=[
        "id",
        "title",
        "genre",
        "description"
    ]
)

train_df = train_df.fillna("")

print("Training data shape:", train_df.shape)


# ============================================================
# 2. LOAD TEST DATA
# ============================================================

test_file = "Genre Classification Dataset/test_data_solution.txt"

test_df = pd.read_csv(
    test_file,
    sep=":::",
    engine="python",
    header=None,
    names=[
        "id",
        "title",
        "genre",
        "description"
    ]
)

test_df = test_df.fillna("")

print("Test data shape:", test_df.shape)


# ============================================================
# 3. CLEAN GENRE LABELS
# ============================================================

train_df["genre"] = (
    train_df["genre"]
    .astype(str)
    .str.strip()
    .str.lower()
)

test_df["genre"] = (
    test_df["genre"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ============================================================
# 4. PREPARE TEXT
# ============================================================

train_df["text"] = (
    train_df["title"].astype(str)
    + " "
    + train_df["description"].astype(str)
)

test_df["text"] = (
    test_df["title"].astype(str)
    + " "
    + test_df["description"].astype(str)
)


X_train = train_df["text"]
y_train = train_df["genre"]

X_test = test_df["text"]
y_test = test_df["genre"]


print("\nNumber of genres:", y_train.nunique())

print("\nGenres:")
print(sorted(y_train.unique()))


# ============================================================
# 5. TF-IDF SETTINGS
# ============================================================

tfidf_settings = {
    "lowercase": True,
    "stop_words": "english",
    "ngram_range": (1, 2),
    "max_features": 200000,
    "min_df": 2,
    "sublinear_tf": True
}


# ============================================================
# 6. NAIVE BAYES
# ============================================================

print("\n========================================")
print("TRAINING NAIVE BAYES")
print("========================================")

naive_bayes_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=200000,
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        MultinomialNB(
            alpha=0.1
        )
    )
])

naive_bayes_model.fit(
    X_train,
    y_train
)

nb_predictions = naive_bayes_model.predict(
    X_test
)

nb_accuracy = accuracy_score(
    y_test,
    nb_predictions
)

print(
    f"Naive Bayes Accuracy: "
    f"{nb_accuracy * 100:.2f}%"
)

print(
    classification_report(
        y_test,
        nb_predictions,
        zero_division=0
    )
)


# ============================================================
# 7. SAVE NAIVE BAYES
# ============================================================

joblib.dump(
    naive_bayes_model,
    "movie_genre_naive_bayes.joblib"
)

print(
    "Saved: movie_genre_naive_bayes.joblib"
)


# ============================================================
# 8. LOGISTIC REGRESSION
# ============================================================

print("\n========================================")
print("TRAINING LOGISTIC REGRESSION")
print("========================================")

logistic_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=200000,
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

logistic_model.fit(
    X_train,
    y_train
)

lr_predictions = logistic_model.predict(
    X_test
)

lr_accuracy = accuracy_score(
    y_test,
    lr_predictions
)

print(
    f"Logistic Regression Accuracy: "
    f"{lr_accuracy * 100:.2f}%"
)

print(
    classification_report(
        y_test,
        lr_predictions,
        zero_division=0
    )
)


# ============================================================
# 9. SAVE LOGISTIC REGRESSION
# ============================================================

joblib.dump(
    logistic_model,
    "movie_genre_logistic_regression.joblib"
)

print(
    "Saved: movie_genre_logistic_regression.joblib"
)


# ============================================================
# 10. SVM
# ============================================================

print("\n========================================")
print("TRAINING SVM")
print("========================================")

svm_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=200000,
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LinearSVC(
            C=2.0,
            class_weight="balanced"
        )
    )
])

svm_model.fit(
    X_train,
    y_train
)

svm_predictions = svm_model.predict(
    X_test
)

svm_accuracy = accuracy_score(
    y_test,
    svm_predictions
)

print(
    f"SVM Accuracy: "
    f"{svm_accuracy * 100:.2f}%"
)

print(
    classification_report(
        y_test,
        svm_predictions,
        zero_division=0
    )
)


# ============================================================
# 11. SAVE SVM
# ============================================================

joblib.dump(
    svm_model,
    "movie_genre_svm.joblib"
)

print(
    "Saved: movie_genre_svm.joblib"
)


# ============================================================
# 12. MODEL COMPARISON
# ============================================================

model_scores = {
    "Naive Bayes": nb_accuracy,
    "Logistic Regression": lr_accuracy,
    "SVM": svm_accuracy
}

print("\n========================================")
print("MODEL COMPARISON")
print("========================================")

for name, score in model_scores.items():

    print(
        f"{name}: {score * 100:.2f}%"
    )


# ============================================================
# 13. SELECT BEST MODEL
# ============================================================

best_model_name = max(
    model_scores,
    key=model_scores.get
)

if best_model_name == "Naive Bayes":

    best_model = naive_bayes_model

elif best_model_name == "Logistic Regression":

    best_model = logistic_model

else:

    best_model = svm_model


print("\n========================================")
print("BEST MODEL")
print("========================================")

print(
    f"Best Model: {best_model_name}"
)

print(
    f"Best Accuracy: "
    f"{model_scores[best_model_name] * 100:.2f}%"
)


# ============================================================
# 14. SAVE BEST MODEL
# ============================================================

joblib.dump(
    best_model,
    "movie_genre_model.joblib"
)

print(
    "Saved: movie_genre_model.joblib"
)


# ============================================================
# 15. SAVE MODEL COMPARISON CSV
# ============================================================

comparison_df = pd.DataFrame({

    "Model": [
        "Naive Bayes",
        "Logistic Regression",
        "SVM"
    ],

    "Accuracy": [
        nb_accuracy,
        lr_accuracy,
        svm_accuracy
    ]
})

comparison_df["Accuracy_Percentage"] = (
    comparison_df["Accuracy"] * 100
)

comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
)

comparison_df.to_csv(
    "model_comparison.csv",
    index=False
)


# ============================================================
# 16. SAVE TEST PREDICTIONS
# ============================================================

prediction_df = pd.DataFrame({

    "id": test_df["id"],

    "title": test_df["title"],

    "actual_genre": y_test,

    "naive_bayes_prediction":
        nb_predictions,

    "logistic_regression_prediction":
        lr_predictions,

    "svm_prediction":
        svm_predictions
})

prediction_df.to_csv(
    "test_predictions.txt",
    sep="\t",
    index=False
)


# ============================================================
# 17. FINISHED
# ============================================================

print("\n========================================")
print("TRAINING COMPLETED")
print("========================================")

print("Generated files:")

print("1. movie_genre_naive_bayes.joblib")
print("2. movie_genre_logistic_regression.joblib")
print("3. movie_genre_svm.joblib")
print("4. movie_genre_model.joblib")
print("5. model_comparison.csv")
print("6. test_predictions.txt")

Training data shape: (54214, 4)
Test data shape: (54200, 4)

Number of genres: 27

Genres:
['action', 'adult', 'adventure', 'animation', 'biography', 'comedy', 'crime', 'documentary', 'drama', 'family', 'fantasy', 'game-show', 'history', 'horror', 'music', 'musical', 'mystery', 'news', 'reality-tv', 'romance', 'sci-fi', 'short', 'sport', 'talk-show', 'thriller', 'war', 'western']

TRAINING NAIVE BAYES
Naive Bayes Accuracy: 53.45%
              precision    recall  f1-score   support

      action       0.72      0.04      0.08      1314
       adult       0.69      0.07      0.13       590
   adventure       0.82      0.09      0.16       775
   animation       0.00      0.00      0.00       498
   biography       0.00      0.00      0.00       264
      comedy       0.55      0.46      0.50      7446
       crime       0.00      0.00      0.00       505
 documentary       0.57      0.90      0.70     13096
       drama       0.47      0.84      0.60     13612
      family       1.00  